In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Install required libraries
!pip install ultralytics roboflow
!git clone https://github.com/ZQPei/deep_sort_pytorch.git
%cd deep_sort_pytorch
!pip install torch torchvision torchaudio
!pip install opencv-python
!pip install scipy
!pip install matplotlib
!pip install filterpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 69.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
Cloning into 'deep_sort_pytorch'...
remote: Enumerating objects: 1078, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (54/5

In [5]:
# Download DeepSORT weights
!pip install gdown
!gdown https://drive.google.com/drive/folders/1xhG0kRH1EX5B9_Iz8gQJb7UNnn_riXi6 --folder

Retrieving folder contents
Processing file 1_qwTWdzT9dWNudpusgKavj_4elGgbkUN ckpt.t7
Processing file 14jLQntkvdzYopte30hKE5BBj2UPPuC6u demo.avi
Processing file 1FJm4hBdw3p0F4NLvRVK2NImWQ3W_8gEa demo2.avi
Processing file 1lfCXBm5ltH-6CjJ1a5rqiZoWgGmRsZSY original_ckpt.t7
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1_qwTWdzT9dWNudpusgKavj_4elGgbkUN
To: /content/deep_sort_pytorch/deepsort_parameters/ckpt.t7
100% 46.0M/46.0M [00:00<00:00, 48.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=14jLQntkvdzYopte30hKE5BBj2UPPuC6u
To: /content/deep_sort_pytorch/deepsort_parameters/demo.avi
100% 56.0M/56.0M [00:01<00:00, 46.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1FJm4hBdw3p0F4NLvRVK2NImWQ3W_8gEa
To: /content/deep_sort_pytorch/deepsort_parameters/demo2.avi
100% 21.8M/21.8M [00:00<00:00, 58.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1lfCXBm5ltH

In [6]:
import cv2
import torch
from ultralytics import YOLO
from roboflow import Roboflow
import numpy as np
import math
from google.colab.patches import cv2_imshow
from deep_sort_pytorch.utils.parser import get_config
from deep_sort_pytorch.deep_sort import DeepSort

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [7]:
# Load YOLOv8 model for object detection
model = YOLO("yolov8n.pt")

In [8]:
# COCO class names
classNames = ["person", "bicycle", "car", "motorbike", "aeroplane", "bus", "train", "truck", "boat",
              "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat",
              "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella",
              "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball", "kite", "baseball bat",
              "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle", "wine glass", "cup",
              "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange", "broccoli",
              "carrot", "hot dog", "pizza", "donut", "cake", "chair", "sofa", "pottedplant", "bed",
              "diningtable", "toilet", "tvmonitor", "laptop", "mouse", "remote", "keyboard", "cell phone",
              "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors",
              "teddy bear", "hair drier", "toothbrush"
              ]

In [9]:
# Load DeepSORT configuration
cfg_deep = get_config()
cfg_deep.merge_from_file("/content/deep_sort_pytorch/configs/deep_sort.yaml")
# Set path to ReID model weights
cfg_deep.DEEPSORT.REID_CKPT = "/content/deep_sort_pytorch/deepsort_parameters/ckpt.t7"

# Initialize DeepSORT tracker
deepsort = DeepSort(cfg_deep.DEEPSORT.REID_CKPT,
                    max_dist=cfg_deep.DEEPSORT.MAX_DIST,
                    min_confidence=0.2,
                    nms_max_overlap=cfg_deep.DEEPSORT.NMS_MAX_OVERLAP,
                    max_iou_distance=cfg_deep.DEEPSORT.MAX_IOU_DISTANCE,
                    max_age=50,
                    n_init=cfg_deep.DEEPSORT.N_INIT,
                    nn_budget=cfg_deep.DEEPSORT.NN_BUDGET,
                    use_cuda=torch.cuda.is_available())

In [10]:
# Function to draw bounding boxes and labels
def draw_boxes(img, bbox, identities = None, categories = None, names = None, offset = (0,0)):
  for i,box in enumerate(bbox):
    x1, y1, x2, y2 = [int(i) for i in box]
    x1 += offset[0]
    x2 += offset[0]
    y1 += offset[1]
    y2 += offset[1]

    # Get class and track ID
    cat = int(categories[i]) if categories is not None else 0
    id = int(identities[i]) if identities is not None else 0
    label = str(id) + ':' + names[cat]

    # Draw bounding box and label
    (w,h),_ = cv2.getTextSize(label,cv2.FONT_HERSHEY_SIMPLEX,0.6,1)
    cv2.rectangle(img,(x1,y1),(x2,y2),(0,255,255),3)
    cv2.rectangle(img,(x1,y1-20),(x1+w,y1),(255,155,30),-1)
    cv2.putText(img, label,(x1,y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6,[255,255,255],2)
  return img

In [12]:
# Load video
cap = cv2.VideoCapture("/content/drive/MyDrive/Colab Notebooks/yolo_nas/yolo_nas_deepsort_video/video/VehiclesEnteringandLeaving.mp4") # Relative Path

frame_width = int(cap.get(3))
frame_height = int(cap.get(4))

# Initialize counting variables
count = 0
counted_ids = set()

# Define counting line position
line_y = frame_height * 3 // 4

# Output video writer
out = cv2.VideoWriter('/content/drive/MyDrive/Colab Notebooks/yolo_nas/yolo_nas_deepsort_video/output.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 10, (frame_width,frame_height)) # Relative Path

# Store previous center positions of tracked objects
prev_centers = {}

while True:
  xywh_bboxs = []
  confs = []
  oids = []
  outputs = []
  ret, frame = cap.read()
  if not ret:
    break

  else:
    # Draw counting line
    cv2.line(frame, (0, line_y), (frame_width, line_y), (0,255,0), 2)

    # Run YOLO detection
    result = list(model.predict(frame, conf = 0.1))[0]

    bbox_xyxys = result.boxes.xyxy.tolist()
    confidences = result.boxes.conf.tolist()
    labels = result.boxes.cls.tolist()


    for bbox_xyxy, confidence, cls in zip(bbox_xyxys, confidences, labels):
      bbox = np.array(bbox_xyxy)
      x1, y1, x2, y2 = bbox[0], bbox[1],bbox[2], bbox[3]
      x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

      # Filter detections (only vehicles)
      if int(cls) not in [2, 3, 5, 7]:  # car, motorcycle, bus, truck
        continue
      classname = int(cls)
      class_name = classNames[classname]

      # Convert to center-based format
      cx, cy = int((x1+x2)/2), int((y1+y2)/2)
      bbox_width = abs(x1-x2)
      bbox_height = abs(y1-y2)
      xcycwh = [cx, cy, bbox_width, bbox_height]
      conf = float(confidence)
      xywh_bboxs.append(xcycwh)
      confs.append(conf)
      oids.append(int(cls))

    # Convert to tensor for DeepSORT
    xywhs = torch.tensor(xywh_bboxs, dtype=torch.float32)
    confss = torch.tensor(confs, dtype=torch.float32)

    # Run tracking
    outputs = deepsort.update(xywhs, confss, oids, frame)
    tracks = outputs[0]

    if tracks is not None and len(tracks) > 0:
      bbox_xyxy = tracks[:, :4]
      identities = tracks[:, -2] # class ID
      object_id = tracks[:, -1] # track ID

      # Draw tracking results
      draw_boxes(frame, bbox_xyxy, object_id, identities, classNames)

      for i, box in enumerate(bbox_xyxy):
        x1, y1, x2, y2 = box
        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)

        track_id = int(object_id[i])

        # Check if object crosses the line
        if track_id in prev_centers:
          prev_cy = prev_centers[track_id]

          # Detect crossing using sign change
          if (prev_cy - line_y) * (cy - line_y) < 0:
            if track_id not in counted_ids:
              counted_ids.add(track_id)
              count += 1

        # Update previous position
        prev_centers[track_id] = cy
        if len(prev_centers) > 1000:
          prev_centers = dict(list(prev_centers.items())[-500:])

      # Display count on frame
      cv2.putText(frame, f'Count: {count}', (50,50),cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    # Save frame to output video
    out.write(frame)

# Release resources
cap.release()
out.release()



0: 384x640 52 cars, 3 buss, 6 trucks, 4 traffic lights, 9.3ms
Speed: 4.3ms preprocess, 9.3ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 51 cars, 2 buss, 6 trucks, 4 traffic lights, 8.7ms
Speed: 2.3ms preprocess, 8.7ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 51 cars, 1 bus, 7 trucks, 7 traffic lights, 13.0ms
Speed: 4.0ms preprocess, 13.0ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 49 cars, 1 bus, 7 trucks, 6 traffic lights, 7.7ms
Speed: 2.1ms preprocess, 7.7ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 48 cars, 5 buss, 4 trucks, 6 traffic lights, 9.7ms
Speed: 6.0ms preprocess, 9.7ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 45 cars, 5 buss, 4 trucks, 6 traffic lights, 19.2ms
Speed: 2.9ms preprocess, 19.2ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 52 cars, 5 buss, 4 trucks, 6 tra